In [1]:
import os
import pandas as pd
import easyocr
from tqdm import tqdm

In [2]:
# 1. Налаштування
FOLDER_PATH = 'GD dataset' # Замініть на свій шлях, наприклад 'imgs/train'
SUPPORTED_FORMATS = ('.png', '.jpg', '.jpeg', '.webp')

# Ініціалізуємо EasyOCR (встановіть потрібні мови, наприклад ['ko', 'en'])
reader = easyocr.Reader(['ko', 'en'], gpu=True)

In [3]:
# 2. Збір даних
data = []

# Отримуємо список усіх файлів у папці
all_files = os.listdir(FOLDER_PATH)
image_files = [f for f in all_files if f.lower().endswith(SUPPORTED_FORMATS)]

print(f"Знайдено зображень у папці: {len(image_files)}")
print("Починаємо обробку...")

Знайдено зображень у папці: 50
Починаємо обробку...


In [ ]:
# Проходимося по кожному фото з індикатором прогресу (tqdm)
for img_name in tqdm(image_files):
    img_path = os.path.join(FOLDER_PATH, img_name)
    
    try:
        # Розпізнаємо текст
        results = reader.readtext(img_path)
        
        # Рахуємо загальну кількість символів (без урахування пробілів)
        total_chars = sum(len(text.replace(" ", "")) for _, text, _ in results)
        
        # Рахуємо кількість окремих текстових блоків (bounding boxes)
        total_boxes = len(results)
        
        # НОВЕ: Витягуємо весь текст і з'єднуємо його через пробіл
        extracted_text = " ".join([text for _, text, _ in results])
        
        # Зберігаємо результати
        data.append({
            'Image_Name': img_name,
            'Total_Characters': total_chars,
            'Text_Boxes': total_boxes,
            'Extracted_Text': extracted_text
        })
        
    except Exception as e:
        print(f"Помилка під час обробки {img_name}: {e}")
        data.append({
            'Image_Name': img_name,
            'Total_Characters': 0,
            'Text_Boxes': 0,
            'Extracted_Text': ""
        })

 28%|██▊       | 14/50 [00:01<00:03,  9.76it/s]

In [ ]:
# 3. Створення DataFrame
df = pd.DataFrame(data)

# Сортуємо від найбільшої кількості символів до найменшої (опціонально)
df = df.sort_values(by='Total_Characters', ascending=False).reset_index(drop=True)

print("\n--- Готово! Перші 5 рядків датафрейму ---")
display(df.head())

df.to_csv('characters_count_analysis.csv', index=False)


--- Готово! Перші 5 рядків датафрейму ---


,Image_Name,Total_Characters,Text_Boxes,Extracted_Text
0,19.png,9,6,와 아 아 야 교해교 와하
1,20.png,5,2,카입_ 라입
2,41.png,5,1,Nolol
3,15.png,4,4,위 이 이 임
4,30.png,4,3,V 얘 HY


In [ ]:
df.iloc[df.Total_Characters > 0]

,Image_Name,Total_Characters,Text_Boxes,Extracted_Text
0,19.png,9,6,와 아 아 야 교해교 와하
1,20.png,5,2,카입_ 라입
2,41.png,5,1,Nolol
3,15.png,4,4,위 이 이 임
4,30.png,4,3,V 얘 HY
5,46.png,4,2,대} 떼데
6,9.png,4,2,저녁 저녁
7,2.png,3,1,Koo
8,18.png,3,2,국 다업
9,14.png,3,3,팬 @ @


In [ ]:
import Levenshtein
import jiwer

def global_cer(all_references, all_hypotheses):
    total_edits = sum(
        Levenshtein.distance(ref, hyp) # Використовуємо швидку C-бібліотеку
        for ref, hyp in zip(all_references, all_hypotheses)
    )
    total_chars = sum(max(len(ref), 1) for ref in all_references)
    return total_edits / total_chars

def global_wer(all_references, all_hypotheses):
    return jiwer.wer(all_references, all_hypotheses)

# --- Заповніть нижче: {image_name: "правильний текст"} ---
with open("ground_truth.txt", "r", encoding='utf-8') as f:
    data = [line.strip() for line in f.readlines() if line.strip() != ""]
    
ground_truth = {}
for i in range(1, 51):
    ground_truth[f"{i}.png"] = data[i-1]

In [ ]:
refs, hyps, rows = [], [], []

for img_name, gt_text in ground_truth.items():
    row = df[df['Image_Name'] == img_name]
    if row.empty:
        continue
    pred = row.iloc[0]['Extracted_Text']
    refs.append(gt_text)
    hyps.append(pred)

    # Per-image для таблиці
    cer_distance = Levenshtein.distance(gt_text, pred)
    img_cer = cer_distance / max(len(gt_text), 1)
    
    if len(gt_text.strip()) == 0:
        img_wer = 1.0 if len(pred.strip()) > 0 else 0.0
    else:
        img_wer = jiwer.wer(gt_text, pred)
        
    rows.append({
        'Image_Name': img_name,
        'GT':         gt_text,
        'Predicted':  pred,
        'CER':        round(img_cer, 4),
        'WER':        round(img_wer, 4),
    })

df_cer = pd.DataFrame(rows)
display(df_cer)


,Image_Name,GT,Predicted,CER,WER
0,1.png,깡,가!,2.0000,1.0
1,2.png,카앙,Koo,1.5000,1.0
2,3.png,쿠궁,1X,1.0000,1.0
3,4.png,삐질,뼈질,0.5000,1.0
4,5.png,쾅,사,1.0000,1.0
5,6.png,퍽 퍼억,썩 폐어,0.7500,1.0
6,7.png,퍽 뻑,뼈 퍽,0.6667,1.0
7,8.png,퍽 콰 악,퍽) @,0.6000,1.0
8,9.png,저벅 저벅,저녁 저녁,0.4000,1.0
9,10.png,퍽,뭐,1.0000,1.0


In [ ]:
# --- Глобальні метрики ---
g_cer = global_cer(refs, hyps)
g_wer = global_wer(refs, hyps)

total_ref_chars = sum(len(r) for r in refs)
total_ref_words = sum(len(r.split()) for r in refs)

print('=' * 52)
print('                     OCR METRICS  ')
print('=' * 52)
print(f'Зображень у GT:        {len(refs)}')
print(f'Всього символів (GT):  {total_ref_chars}')
print(f'Всього слів (GT):      {total_ref_words}')
print('-' * 52)
print(f'Global CER:  {g_cer:.4f}  ({g_cer*100:.2f}%)')
print(f'Global WER:  {g_wer:.4f}  ({g_wer*100:.2f}%)')
print('-' * 52)
print(f'Mean CER:  {df_cer["CER"].mean():.4f}  ({df_cer["CER"].mean()*100:.2f}%)')
print(f'Mean WER:  {df_cer["WER"].mean():.4f}  ({df_cer["WER"].mean()*100:.2f}%)')
print('=' * 52)
df_cer.to_csv('ocr_cer_wer_per_image.csv', index=False)

                     OCR METRICS  
Зображень у GT:        50
Всього символів (GT):  138
Всього слів (GT):      71
----------------------------------------------------
Global CER:  0.9710  (97.10%)
Global WER:  1.1127  (111.27%)
----------------------------------------------------
Mean CER:  1.0892  (108.92%)
Mean WER:  1.1500  (115.00%)
